# Lab 20: Haar Wavelets — Local Patterns, Jumps, and Multiscale Structure

This lab is a detailed computational companion to Chapter 20.

Fourier analysis describes a signal using global waves. Haar wavelets describe a signal using local averages and differences. In this lab, you will build the Haar transform from scratch, visualize its basis vectors, use it for compression and denoising, compare it with Fourier ideas, and finish with a two-dimensional image example.

## 0. Imports

We use only standard scientific Python libraries. All examples are synthetic, so the notebook runs without downloading data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. The smallest Haar transform

For two numbers $a,b$, the normalized Haar transform produces

$$s=rac{a+b}{\sqrt{2}}, \qquad d=rac{a-b}{\sqrt{2}}.$$

The first coefficient is an average-like coordinate. The second coefficient is a difference-like coordinate.

In [ ]:
H2 = np.array([[1, 1], [1, -1]], dtype=float) / np.sqrt(2)
x = np.array([20, 220])
c = H2 @ x
x_reconstructed = H2.T @ c

print("H2 =")
print(H2)
print("Signal x =", x)
print("Haar coefficients [average-like, difference-like] =", c)
print("Reconstructed signal =", x_reconstructed)
print("Energy in x:", np.linalg.norm(x)**2)
print("Energy in coefficients:", np.linalg.norm(c)**2)

## 2. Build Haar matrices recursively

The Haar matrix for length $n=2^m$ can be built recursively.
The top half computes coarse averages using a smaller Haar matrix.
The bottom half computes local pair differences.

In [ ]:
def haar_matrix(n):
    """Return an orthonormal Haar matrix for n = 2^m."""
    if n < 1 or (n & (n - 1)) != 0:
        raise ValueError("n must be a power of 2")
    if n == 1:
        return np.array([[1.0]])
    H_small = haar_matrix(n // 2)
    top = np.kron(H_small, [1, 1]) / np.sqrt(2)
    bottom = np.kron(np.eye(n // 2), [1, -1]) / np.sqrt(2)
    return np.vstack([top, bottom])

for n in [2, 4, 8]:
    H = haar_matrix(n)
    print(f"n = {n}, orthogonality error =", np.linalg.norm(H @ H.T - np.eye(n)))
    print(H)
    print()

## 3. Visualize Haar basis vectors

Each row of the Haar matrix is a basis vector. The first row is global. Later rows become increasingly local.

In [ ]:
def plot_basis(H, title):
    n = H.shape[0]
    fig, axes = plt.subplots(n, 1, figsize=(10, 1.2*n), sharex=True)
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.step(np.arange(n), H[i], where='mid')
        ax.axhline(0, linewidth=0.8)
        ax.set_ylabel(f'h{i}')
    axes[-1].set_xlabel('signal position')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

H16 = haar_matrix(16)
plot_basis(H16[:8], 'First 8 Haar basis vectors for length 16')

## 4. Multiscale coefficients of simple signals

We compare three signals:

1. a constant signal,
2. a step signal,
3. an alternating signal.

The location of large coefficients tells us the scale of the structure.

In [ ]:
n = 32
H = haar_matrix(n)
signals = {
    'constant': np.ones(n),
    'step': np.r_[np.ones(n//2), 4*np.ones(n//2)],
    'alternating': np.array([1 if i % 2 == 0 else -1 for i in range(n)], dtype=float),
}

for name, x in signals.items():
    c = H @ x
    plt.figure(figsize=(10, 3))
    plt.stem(np.arange(n), c)
    plt.title(f'Haar coefficients: {name}')
    plt.xlabel('coefficient index')
    plt.ylabel('coefficient value')
    plt.show()

## 5. Compression by keeping the largest coefficients

Because the Haar transform is orthonormal, keeping the largest coefficients is a natural compression strategy. For piecewise constant signals, this often works very well.

In [ ]:
def keep_largest(c, k):
    out = np.zeros_like(c)
    idx = np.argsort(np.abs(c))[-k:]
    out[idx] = c[idx]
    return out

n = 128
H = haar_matrix(n)
x = np.r_[
    1*np.ones(20),
    3*np.ones(30),
    -1*np.ones(18),
    2*np.ones(40),
    0.5*np.ones(20)
]
x = x[:n]

c = H @ x
for k in [4, 8, 16, 32]:
    x_hat = H.T @ keep_largest(c, k)
    err = np.linalg.norm(x - x_hat) / np.linalg.norm(x)
    plt.figure(figsize=(10, 3))
    plt.plot(x, label='original')
    plt.plot(x_hat, label=f'k={k}, relative error={err:.3f}')
    plt.legend()
    plt.title('Haar compression of a piecewise constant signal')
    plt.show()

## 6. Denoising by thresholding

A noisy step signal often has many small Haar coefficients caused by noise. Thresholding removes small coefficients and keeps strong structural changes.

In [ ]:
n = 128
H = haar_matrix(n)
clean = np.r_[np.zeros(32), 2*np.ones(32), -1*np.ones(32), 1.2*np.ones(32)]
noisy = clean + 0.35*rng.normal(size=n)
coeff = H @ noisy

for tau in [0.2, 0.5, 0.9]:
    coeff_thr = coeff.copy()
    coeff_thr[np.abs(coeff_thr) < tau] = 0
    denoised = H.T @ coeff_thr
    plt.figure(figsize=(10, 3))
    plt.plot(clean, label='clean')
    plt.plot(noisy, alpha=0.55, label='noisy')
    plt.plot(denoised, linewidth=2, label=f'threshold {tau}')
    plt.legend()
    plt.title('Haar thresholding for denoising')
    plt.show()

## 7. Fourier versus Haar

Fourier is efficient for smooth periodic signals. Haar is efficient for local jumps and piecewise constant signals.

This section compares the magnitudes of coefficients for a sine wave and a step signal.

In [ ]:
n = 128
t = np.arange(n)
H = haar_matrix(n)

smooth = np.sin(2*np.pi*5*t/n)
step = np.r_[np.ones(n//2), -np.ones(n//2)]

for name, x in [('smooth sine wave', smooth), ('step signal', step)]:
    haar_c = H @ x
    fft_c = np.fft.fft(x) / np.sqrt(n)
    plt.figure(figsize=(10, 3))
    plt.plot(np.sort(np.abs(haar_c))[::-1], marker='o', label='sorted Haar magnitudes')
    plt.plot(np.sort(np.abs(fft_c))[::-1], marker='s', label='sorted Fourier magnitudes')
    plt.yscale('log')
    plt.title(f'Coefficient decay: {name}')
    plt.xlabel('rank after sorting by magnitude')
    plt.ylabel('coefficient magnitude, log scale')
    plt.legend()
    plt.show()

## 8. Two-dimensional Haar transform for images

For an image matrix $X$, a simple 2D Haar transform is

$$C = H X H^T.$$

The upper-left coefficients contain coarse information. Other regions capture horizontal, vertical, and diagonal changes.

In [ ]:
def make_synthetic_image(n=64):
    Y, X = np.mgrid[0:n, 0:n]
    img = np.zeros((n, n))
    img += 0.2
    img[(X > 10) & (X < 28) & (Y > 12) & (Y < 48)] = 0.75
    img[(X-45)**2 + (Y-32)**2 < 12**2] = 1.0
    img[Y > X + 10] += 0.15
    return np.clip(img, 0, 1)

n = 64
img = make_synthetic_image(n)
H = haar_matrix(n)
C = H @ img @ H.T

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title('Synthetic image')
plt.axis('off')
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(np.log1p(np.abs(C)), cmap='gray')
plt.title('Log magnitude of 2D Haar coefficients')
plt.axis('off')
plt.show()

## 9. Image compression with 2D Haar coefficients

Now keep only the largest 2D Haar coefficients and reconstruct.

In [ ]:
def keep_largest_2d(C, k):
    flat = C.ravel()
    out = np.zeros_like(flat)
    idx = np.argsort(np.abs(flat))[-k:]
    out[idx] = flat[idx]
    return out.reshape(C.shape)

for k in [64, 128, 256, 512]:
    Ck = keep_largest_2d(C, k)
    rec = H.T @ Ck @ H
    err = np.linalg.norm(img - rec) / np.linalg.norm(img)
    plt.figure(figsize=(5, 5))
    plt.imshow(np.clip(rec, 0, 1), cmap='gray')
    plt.title(f'Keep {k} / {n*n} coefficients, rel. error {err:.3f}')
    plt.axis('off')
    plt.show()

## 10. Student exploration

Try the following experiments.

1. Change the synthetic image by adding more rectangles or circles.
2. Add Gaussian noise to the image and threshold small Haar coefficients.
3. Compare Haar compression with SVD compression from Chapter 17.
4. Compare Haar coefficients of a smooth gradient image and a sharp-edge image.
5. Explain in words which type of image structure Haar represents efficiently.

In [ ]:
# Workspace for your experiments
# Example: add noise and threshold 2D Haar coefficients
noise_level = 0.15
noisy_img = np.clip(img + noise_level*rng.normal(size=img.shape), 0, 1)
C_noisy = H @ noisy_img @ H.T
threshold = 0.25
C_denoised = C_noisy.copy()
C_denoised[np.abs(C_denoised) < threshold] = 0
img_denoised = H.T @ C_denoised @ H

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes, [img, noisy_img, img_denoised], ['clean', 'noisy', 'Haar denoised']):
    ax.imshow(np.clip(data, 0, 1), cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.show()

## Lab summary

In this lab you built the Haar transform as an orthonormal matrix, interpreted its coefficients as multiscale averages and differences, used it for compression and denoising, compared it with Fourier coefficients, and applied a two-dimensional Haar transform to images.

The main lesson is that a basis is a language. Haar is the language of local change.